**Canada.ca**

This scrapes the government websites and gives updates as required

In [2]:
import os

from dotenv import load_dotenv
from streamed_scraper import fetch_website_contents, fetch_website_contents_playwright
from IPython.display import Markdown, display
from openai import OpenAI


In [4]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
api_key_llama = os.getenv('anything')


In [12]:
openai = OpenAI()
GPT_4_MODEL = "gpt-4.1-mini"
GPT_5_MODEL = "gpt-5-nano"
LLAMA_MODEL = "llama3.2"

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

system_prompt = """
You extracted content from a webpage. and give a summary of the content, paying attention to details. 
1. If the page is telling a story, give a summary of the story.
2. If the page is telling history, give a summary of the history, not just the headers but the story.
2. If the page is a list of items, give a summary of the items.
3. If the page is a list of links, give a summary of the links.
4. If the page is a list of images, give a summary of the images.
5. If the page is a list of videos, give a summary of the videos.
6. If the page is a list of audio, give a summary of the audio.
7. If the page is a list of documents, give a summary of the documents.
8. If the page is telling the news, give a summary of the news.
9. If the page is telling the weather, give a summary of the weather.
10. If the page is telling the time, give a summary of the time.
11. If the page is telling the date, give a summary of the date.
12. If the page is telling the weather, give a summary of the weather.
13. If the page is telling the time, give a summary of the time.
14. If the page is telling the date, give a summary of the date.
15. If the page is a job posting, give the job posting.
16. If the page is a job search, give the job search.
17. If the page is a job listing, give the job listing.
18. If the page is a job application, give a summary of the job application.
19. If the page is a job interview, give a summary of the job interview.
20. If the page is a job offer, give a summary of the job offer.
21. If the page is a job site, list all the jobs available.
"""

user_prompt_prefix = """
Make your response concise and to the point, as comprehensive as possible
"""

def pass_prompts(website_text: str):
    return [
        {"role": "system", "content": system_prompt},
         {
            "role": "user",
            "content": f"""
{user_prompt_prefix} 
Webpage text:
{website_text}
"""
        },
    ]

def get_gpt_response(messages, model=LLAMA_MODEL):
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        # temperature=0.5,
        # max_tokens=1000,
        # top_p=1,
        # frequency_penalty=0,
        # presence_penalty=0
    )
    return response.choices[0].message.content

In [13]:

async def get_page_text(url):
    try:
        text = fetch_website_contents(url)

        if len(text.strip()) < 300:
            raise ValueError("Too little content from static scrape")

        return text
    except Exception:
        return await fetch_website_contents_playwright(url)


async def get_summary(website_url):
    website_text = await get_page_text(website_url)

    print("SCRAPED PREVIEW:")
    print(website_text[:3000])

    response = openai.chat.completions.create(
        model=GPT_4_MODEL,
        messages=pass_prompts(website_text),
    )
    return response.choices[0].message.content


async def display_summary(website_url):
    summary = await get_summary(website_url)
    display(Markdown(summary))

In [14]:
await display_summary("https://www.jobbank.gc.ca/jobsearch/jobsearch?fskl=100000&fskl=15141&page=1&sort=M")

SCRAPED PREVIEW:
Available jobs - Search - Job Bank

Skip to job search
Skip to main content
Skip to "About this Web application"
Language selection
Français
fr
Government of Canada /
Gouvernement du Canada
Search
Search website
Search
Job Bank
Job Bank
Account menu
Sign in
Job seekers
Employers
Menu and search
Menu
Menu
Account menu
Sign in
Job seekers
Employers
Main navigation menu
Job search
Training and careers
Labour market information
Hiring
Help
About
You are here:
Job Bank
Dashboard
Job Search
Keywords:
Location:
All of Canada
Current location
Search
Advanced
Browse
Search
Loading, please wait...
Job Search Mobile
Available jobs - Search
Skip to filters
Below is an interactive map
Skip to map
Filters
Sort by
Search
Recent searches
Available jobs
Create alert
Remove keyword
Remote
Remove keyword
Hybrid
813
results
Sort by:
Best match
Date posted
New
Hybrid
Direct Apply
Posted on Job Bank
This job was posted directly by the employer on Job Bank.
Job Bank
social media coordinator


The webpage is from the Government of Canada's Job Bank, listing available jobs across Canada with filters for location and job type. It features recently posted job openings with details such as position title, employer, location, salary, posting date, and job type (e.g., hybrid, direct apply). Examples include:

- Social Media Coordinator at Level Up Enterprises, Vancouver, BC, paying $36.60/hour.
- International Program Director (Cooperative) at Alinea International Ltd., Calgary, AB, salary $110,000-$120,000 annually (negotiable).
- Senior Bookkeeper at Sixtop, Ottawa, ON.

Users can sign in or create an account to save favorites and apply. The platform supports job search, alerts, and account management.